In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"

In [ ]:
df = pd.read_csv(f'{DATA_DIR}/v1/function_dataset_cleaned.csv')

In [ ]:
X = df.drop(columns=['label'])
y = df['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10, stratify=y)

## 特徵工程

In [ ]:
# 先將原本100個欄位拿去訓練模型
best_model = joblib.load(f'{MODEL_DIR}/v1/best_model.joblib')
best_model.fit(X_train, y_train)

In [ ]:
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
print("原始特徵:")
print(f"準確度: {accuracy:.4f}")
print(f"混淆矩陣:\n{confusion_matrix(y_test, y_pred)}\n")
print(f"分類報告:\n{classification_report(y_test, y_pred)}")

In [ ]:
# 將原始資料標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 訓練集

In [ ]:
train_first_diff = np.diff(X_train.values, axis=1)          # 一階差分
train_second_diff = np.diff(X_train.values, n=2, axis=1)    # 二階差分
train_amplitude = X_train.max(axis=1) - X_train.min(axis=1) # 變化總幅度

train_x = np.linspace(-2, 2, X_train.shape[1]).reshape(-1, 1)
train_slopes = []                                           # 每一筆函數的整體迴歸斜率
for row in X_train.values:
    train_model = LinearRegression()
    train_model.fit(train_x, row)
    train_slopes.append(train_model.coef_[0])
train_slopes = np.array(train_slopes)

In [ ]:
print(X_train_scaled.shape)     # 訓練數據的形狀

In [ ]:
print(f"原始最大值:\n{X_train.max(axis=0)}\n")              # 最大值
print(f"標準化後的最大值:\n{X_train_scaled.max(axis=0)}\n")

In [ ]:
print(f"原始最小值:\n{X_train.min(axis=0)}\n")              # 最小值
print(f"標準化後的最小值:\n{X_train_scaled.min(axis=0)}\n")

In [ ]:
print(X_train.mean(axis=0))     # 原始平均值

In [ ]:
print(X_train.std(axis=0))      # 原始標準差

In [ ]:
print(train_slopes[:5])         # 迴歸斜率(取前5筆資料)

In [ ]:
print("經過一階差分處理後的結果:(取前5筆資料)")
train_first_diff_mean = train_first_diff.mean(axis=1)
train_first_diff_std = train_first_diff.std(axis=1)
print(f"平均值:\n{train_first_diff_mean[:5]}")
print(f"標準差:\n{train_first_diff_std[:5]}\n")

In [ ]:
print("經過二階差分處理後的結果:(取前5筆資料)")
train_second_diff_mean = train_second_diff.mean(axis=1)
train_second_diff_std = train_second_diff.std(axis=1)
print(f"平均值:\n{train_second_diff_mean[:5]}")
print(f"標準差:\n{train_second_diff_std[:5]}\n")

In [ ]:
print(train_amplitude[:5])      # 變化幅度:(取前5筆資料)

In [ ]:
# 建立 X_train 的6個數學特徵(斜率, 總變化量, 一階差分, 二階差分)
train_math_features = pd.DataFrame({'slope': train_slopes,
                                    'amplitude': train_amplitude,
                                    'first_diff_mean': train_first_diff_mean,
                                    'first_diff_std': train_first_diff_std,
                                    'second_diff_mean': train_second_diff_mean,
                                    'second_diff_std': train_second_diff_std})

In [ ]:
print(train_math_features.head())       # X_train數學特徵前5筆

In [ ]:
# 加入原始特徵
X_train_math = pd.concat([X_train.reset_index(drop=True),
                          train_math_features.reset_index(drop=True)], axis=1)

In [ ]:
print(f"X_train_math 形狀: {X_train_math.shape}")

### 測試集

In [ ]:
test_first_diff = np.diff(X_test.values, axis=1)            # 一階差分
test_second_diff = np.diff(X_test.values, n=2, axis=1)      # 二階差分
test_amplitude = X_test.max(axis=1) - X_test.min(axis=1)    # 變化總幅度

test_x = np.linspace(-2, 2, X_test.shape[1]).reshape(-1, 1)
test_slopes = []                                            # 每一筆函數的整體迴歸斜率
for row in X_test.values:
    test_model = LinearRegression()
    test_model.fit(test_x, row)
    test_slopes.append(test_model.coef_[0])
test_slopes = np.array(test_slopes)

# 一階差分統計
test_first_diff_mean = test_first_diff.mean(axis=1)
test_first_diff_std = test_first_diff.std(axis=1)
# 二階差分統計
test_second_diff_mean = test_second_diff.mean(axis=1)
test_second_diff_std = test_second_diff.std(axis=1)

In [ ]:
# 建立 X_test 的6個數學特徵(斜率, 總變化量, 一階差分, 二階差分)
test_math_features = pd.DataFrame({'slope': test_slopes,
                                   'amplitude': test_amplitude,
                                   'first_diff_mean': test_first_diff_mean,
                                   'first_diff_std': test_first_diff_std,
                                   'second_diff_mean': test_second_diff_mean,
                                   'second_diff_std': test_second_diff_std})

In [ ]:
print(test_math_features.head())        # X_test數學特徵前5筆

In [ ]:
# 加入原始特徵
X_test_math = pd.concat([X_test.reset_index(drop=True),
                         test_math_features.reset_index(drop=True)], axis=1)

In [ ]:
print(f"X_test_math 形狀: {X_test_math.shape}")

### 把訓練集跟測試集結合成一個大表格

In [ ]:
X_math_merged = pd.concat([X_train_math.reset_index(drop=True),
                           X_test_math.reset_index(drop=True)], axis=0)

In [ ]:
print(X_math_merged.shape)      # 合併之後的形狀

### 將合併後的表格儲存成新的檔案

In [ ]:
y_merged = pd.concat([y_train.reset_index(drop=True),
                      y_test.reset_index(drop=True)], axis=0)

In [ ]:
math_merged = pd.concat([y_merged, X_math_merged], axis=1)

In [ ]:
math_merged.to_csv(f'{DATA_DIR}/v1/function_dataset_reset.csv', index=False)

### 標準化

In [ ]:
scaler_after = StandardScaler()
X_train_math_scaled = scaler_after.fit_transform(X_train_math)
X_test_math_scaled = scaler_after.transform(X_test_math)

In [ ]:
print("標準化之後:")
print(f"平均值:\n{X_train_math_scaled.mean(axis=0)}")
print(f"標準差:\n{X_train_math_scaled.std(axis=0)}")

### 將結合後的數據再訓練模型一次

In [ ]:
best_model_math = joblib.load(f'{MODEL_DIR}/v1/best_model.joblib')
best_model_math.fit(X_train_math, y_train.reset_index(drop=True))

In [ ]:
y_pred_math = best_model_math.predict(X_test_math)
math_accuracy = accuracy_score(y_test.reset_index(drop=True), y_pred_math)

In [ ]:
print("新增一些數學特徵之後:")
print(f"準確度: {math_accuracy:.4f}")
print(f"混淆矩陣:\n{confusion_matrix(y_test.reset_index(drop=True), y_pred_math)}\n")
print(f"分類報告:\n{classification_report(y_test.reset_index(drop=True), y_pred_math)}")

### 比較結果

In [ ]:
# 實驗結果
print(f"原始特徵準確度: {accuracy:.3f}")
print(f"數學特徵準確度: {math_accuracy:.3f}")
improvement = math_accuracy - accuracy
print(f"比原本特徵進步了: {improvement:+.3f}")
print("結論:", end=" ")
if improvement > 0:
    print("加入數學特徵後, 準確度提升。")
elif improvement < 0:
    print("加入數學特徵後, 準確度下降。")
else:
    print("加入數學特徵後, 準確度沒有改變。")

## 特徵重要性

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Microsoft JhengHei"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
feature_importance = best_model_math.feature_importances_
feature_names = X_train_math.columns
importance_df = pd.DataFrame({'Feature': feature_names,
                              'Importance': feature_importance})

In [ ]:
# 由高到低排序
importance_df = importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

In [ ]:
print(importance_df.head(20).to_string(index=False, formatters={'Importance': '{:.4f}'.format}))

In [ ]:
importance_df.to_html(f'{DATA_DIR}/v1/feature_importance.html')

In [ ]:
math_feature_names = ['slope', 'amplitude', 'first_diff_mean', 'first_diff_std',
                      'second_diff_mean', 'second_diff_std']
math_importance_df = importance_df[importance_df['Feature'].isin(math_feature_names)].copy()

In [ ]:
print(math_importance_df.to_string(index=False, formatters={'Importance': '{:.4f}'.format}))

In [ ]:
math_total_importance = (math_importance_df['Importance'].sum())    # 計算數學特徵的重要性總和
raw_total_importance = (importance_df[~importance_df['Feature'].isin(math_feature_names)]['Importance'].sum())
print("特徵群組重要性總和:")
print(f"原始 y: {raw_total_importance:.3f}")
print(f"6 個數學特徵: {math_total_importance:.3f}")

In [ ]:
most_important_math = (math_importance_df.iloc[0])      # 找出最重要的數學特徵
print(f"最重要的數學特徵: {most_important_math['Feature']} ({most_important_math['Importance']:.4f})")

In [ ]:
# 畫出前 20 名 Feature Importance
top_n = 20
top_features = importance_df.head(top_n).copy()
top_features = top_features.sort_values(by='Importance', ascending=True)    # 為了畫圖讓最重要的在最上面

In [ ]:
# 繪製圖表
plt.figure(figsize=(10, 6))
plt.barh(top_features['Feature'], top_features['Importance'])
plt.xlabel("重要程度")
plt.ylabel("特徵")
plt.title("最佳模型的Feature Importance前20名")
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.legend()
plt.savefig(f'{DATA_DIR}/v1/feature_importance_results.png')
plt.show()

## 模型調參

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [ ]:
# 調參前模型
baseline_model = joblib.load(f'{MODEL_DIR}/v1/best_model.joblib')
baseline_model.fit(X_train_math, y_train.reset_index(drop=True))

In [ ]:
baseline_train_pred = baseline_model.predict(X_train_math)
baseline_test_pred = baseline_model.predict(X_test_math)
baseline_train_acc = accuracy_score(y_train.reset_index(drop=True), baseline_train_pred)
baseline_test_acc = accuracy_score(y_test.reset_index(drop=True), baseline_test_pred)
print("模型調參前")
print(f"訓練數據準確度: {baseline_train_acc:.4f}")
print(f"測試數據準確度: {baseline_test_acc:.4f}")

In [ ]:
# 設定 GridSearchCV
rf_model = RandomForestClassifier(random_state=10)
param_grid = {'n_estimators': [100, 150, 200],
              'max_depth': [None, 10, 20],
              'min_samples_split': [2, 3, 5],
              'min_samples_leaf': [1]}

In [ ]:
# 開始 GridSearchCV
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_math, y_train.reset_index(drop=True))

In [ ]:
print(grid_search.best_params_)     # 最佳參數

In [ ]:
print(f"最佳CV準確度: {grid_search.best_score_:.4f}")

In [ ]:
# 使用最佳模型預測
best_model = grid_search.best_estimator_

In [ ]:
train_pred = best_model.predict(X_train_math)
test_pred = best_model.predict(X_test_math)
train_accuracy = accuracy_score(y_train.reset_index(drop=True), train_pred)
test_accuracy = accuracy_score(y_test.reset_index(drop=True), test_pred)
print("模型調參後")
print(f"訓練數據準確度: {train_accuracy:.4f}")
print(f"測試數據準確度: {test_accuracy:.4f}")

In [ ]:
# 比較調參前後
print(f"調參前準確度: {baseline_test_acc:.4f}")
print(f"調參後準確度: {test_accuracy:.4f}")
improvement = test_accuracy - baseline_test_acc
print(f"改善幅度: {improvement:+.4f}")
print("結論:", end=" ")
if improvement > 0:
    print("模型調參後，測試準確度提升。")
elif improvement < 0:
    print("模型調參後，測試準確度下降。")
else:
    print("測試準確度沒有改變。\n")

In [ ]:
print(f"混淆矩陣:\n{confusion_matrix(y_test.reset_index(drop=True), test_pred)}")

In [ ]:
print(f"分類報告:\n{classification_report(y_test.reset_index(drop=True), test_pred)}")

In [ ]:
# 儲存最佳模型
joblib.dump(best_model, f'{MODEL_DIR}/v1/best_model_tuned.joblib')

## Ensemble

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline

In [ ]:
tuned = joblib.load(f'{MODEL_DIR}/v1/best_model_tuned.joblib')
tuned.fit(X_train, y_train)

In [ ]:
tuned_train_pred = tuned.predict(X_train)
tuned_test_pred = tuned.predict(X_test)
tuned_train_acc = accuracy_score(y_train, tuned_train_pred)
tuned_test_acc = accuracy_score(y_test, tuned_test_pred)

In [ ]:
# 定義學習器
base_learners = [('DecisionTree', DecisionTreeClassifier(random_state=10)),
                 ('RandomForest', tuned)]
final_estimator = make_pipeline(StandardScaler(), 
                                SVC(random_state=10))

In [ ]:
# 建立 Stacking Ensemble 模型
st_model = StackingClassifier(estimators=base_learners, final_estimator=final_estimator)
st_model.fit(X_train, y_train)

In [ ]:
ensemble_train_pred = st_model.predict(X_train)
ensemble_test_pred = st_model.predict(X_test)
ensemble_train_acc = accuracy_score(y_train, ensemble_train_pred)
ensemble_test_acc = accuracy_score(y_test, ensemble_test_pred)

In [ ]:
print(f"訓練數據準確度: {ensemble_train_acc:.4f}")
print(f"測試數據準確度: {ensemble_test_acc:.4f}\n")

In [ ]:
print(f"混淆矩陣:\n{confusion_matrix(y_test, ensemble_test_pred)}")

In [ ]:
print(f"分類報告:\n{classification_report(y_test, ensemble_test_pred)}")

In [ ]:
# 實驗結果
print(f"模型調參前: {baseline_test_acc:.4f}")
print(f"模型調參後: {tuned_test_acc:.4f}")
print(f"Stacking Ensemble: {ensemble_test_acc:.4f}\n")

In [ ]:
model_results = {'模型調參前': (baseline_test_acc, baseline_model),
                 '模型調參後': (tuned_test_acc, tuned),
                 'Stacking Ensemble': (ensemble_test_acc, st_model)}

In [ ]:
best_model_name = max(model_results, key=lambda name: model_results[name][0])
best_accuracy, final_model = model_results[best_model_name]

In [ ]:
print(f"最佳模型: {best_model_name} 模型")
print(f"準確度: {best_accuracy:.4f}\n")

In [ ]:
# 儲存最終模型
model_package = {'model': final_model,
                 'feature_names': X_train.columns.tolist()}
joblib.dump(model_package, f'{MODEL_DIR}/v1/final_model.joblib')